<a href="https://colab.research.google.com/github/srijakothakonda/YOLOv8-Object-Detection-with-Text-to-Speech/blob/main/Streamlit_genai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ✅ PASTE THIS ENTIRE CELL AND RUN

# Step 1: Install dependencies
!pip install streamlit ultralytics gtts -q

# Step 2: Write the Streamlit app
app_code = """
import streamlit as st
import cv2
import numpy as np
from PIL import Image
from ultralytics import YOLO
from gtts import gTTS
import tempfile
import os

st.set_page_config(page_title="YOLOv8 Object Detection + TTS", layout="wide")
st.title("YOLOv8 Object Detection with Text-to-Speech")
st.markdown("---")

@st.cache_resource
def load_model():
    return YOLO("yolov8n.pt")

model = load_model()
st.success("YOLOv8 Nano model loaded!")

uploaded_file = st.file_uploader("Upload an image", type=["jpg", "jpeg", "png"])

if uploaded_file:
    col1, col2 = st.columns(2)
    image = Image.open(uploaded_file).convert("RGB")
    image_np = np.array(image)
    image_bgr = cv2.cvtColor(image_np, cv2.COLOR_RGB2BGR)

    with col1:
        st.subheader("Original Image")
        st.image(image, use_container_width=True)

    with st.spinner("Running detection..."):
        results = model(image_bgr)

    output_img = cv2.cvtColor(image_bgr.copy(), cv2.COLOR_BGR2RGB)
    detected_labels = []

    for result in results:
        for box in result.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            class_id = int(box.cls[0])
            conf = float(box.conf[0])
            label = model.names[class_id]
            detected_labels.append(label)
            cv2.rectangle(output_img, (x1, y1), (x2, y2), (0, 255, 0), 3)
            cv2.putText(output_img, f"{label} {conf:.2f}", (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

    with col2:
        st.subheader("Detected Objects")
        st.image(output_img, use_container_width=True)

    st.markdown("---")
    unique_labels = list(set(detected_labels))

    if unique_labels:
        st.subheader("Detection Results")
        cols = st.columns(min(len(unique_labels), 4))
        for i, lbl in enumerate(unique_labels):
            cols[i % 4].metric(label="Object", value=lbl, delta=f"{detected_labels.count(lbl)}x")

        speech_text = "Detected objects are: " + ", ".join(unique_labels)
        st.info(f"Speech: {speech_text}")

        with st.spinner("Generating speech..."):
            tts = gTTS(speech_text, lang="en")
            with tempfile.NamedTemporaryFile(delete=False, suffix=".mp3") as f:
                tts.save(f.name)
                with open(f.name, "rb") as audio_file:
                    audio_bytes = audio_file.read()
            os.unlink(f.name)

        st.audio(audio_bytes, format="audio/mp3")
    else:
        st.warning("No objects detected.")
"""

with open("app.py", "w") as f:
    f.write(app_code)
print("app.py written!")

# Step 3: Start Streamlit in background
import subprocess, time, threading, re

proc = subprocess.Popen(
    ["streamlit", "run", "app.py",
     "--server.port=8501",
     "--server.headless=true",
     "--server.enableCORS=false",
     "--server.enableXsrfProtection=false"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(6)
print("Streamlit started!")

# Step 4: Download cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
print("cloudflared ready!")

# Step 5: Start tunnel and capture URL (FIXED)
import sys

tunnel_proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8501"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT  # merge stderr into stdout
)

url_found = False
print("Waiting for tunnel URL...")

for line in tunnel_proc.stdout:
    line = line.decode("utf-8", errors="ignore").strip()
    match = re.search(r'https://[a-zA-Z0-9\-]+\.trycloudflare\.com', line)
    if match:
        url = match.group(0)
        print(f"\n{'='*50}")
        print(f"YOUR APP URL: {url}")
        print(f"{'='*50}\n")
        url_found = True
        break

if not url_found:
    print("Could not find tunnel URL. Try re-running.")

app.py written!
Streamlit started!
cloudflared: Text file busy
cloudflared ready!
Waiting for tunnel URL...

YOUR APP URL: https://amended-toronto-bryant-targeted.trycloudflare.com

